## Tools-ToolNode 工具节点
ToolNode是一个预构建节点，用于在 LangGraph 工作流中执行工具。它能够自动处理并行工具执行、错误处理和状态注入。
### 基本用法

In [1]:
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

# Create the ToolNode with your tools
tool_node = ToolNode([search, calculator])

# Use in a graph
builder = StateGraph(MessagesState)
builder.add_node("tools", tool_node)
# ... add other nodes and edges

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### 工具返回值
可以为工具选择不同的返回值：

- 返回一个string人类可读的结果。
- 返回object模型应解析的结构化结果。
- Command当需要写入状态时，返回一个带有可选消息的 a。

#### 返回字符串
当工具需要提供纯文本供模型读取并在其下一个响应中使用时，返回一个字符串。

In [ ]:
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is currently sunny in {city}."

行为：

- 返回值被转换为`ToolMessage`.
- 模型识别到该文本后，会决定下一步该做什么。
- 除非模型或其他工具稍后进行更改，否则不会更改任何代理状态字段。

当结果为自然可读的文本时，请使用此选项。

#### 返回一个对象
dict当你的工具生成模型应该检查的结构化数据时，返回一个对象（例如，一个）。

In [2]:
from langchain.tools import tool


@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

行为：

- 该对象被序列化并作为工具输出发送回去。
- 该模型可以读取特定字段并对其进行推理。
- 与字符串返回一样，这不会直接更新图状态。

当下游推理受益于显式字段而不是自由格式文本时，请使用此功能。

#### 返回 Command
[`Command`](https://reference.langchain.com/python/langgraph/types/Command)当工具需要更新图状态（例如，设置用户偏好或应用状态）时，返回一个更新。您可以返回`Command`包含或不包含更新参数的更新`ToolMessage`。如果模型需要确认工具操作成功（例如，确认偏好设置更改），请`ToolMessage`在更新中包含一个更新，并使用`is``runtime.tool_call_id`作为`tool_call_id`参数。

In [3]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Set the preferred response language."""
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Language set to {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

行为：

- 该命令使用 . 更新状态`update`。
- 更新后的状态可供同一运行中的后续步骤使用。
- 对于可能被并行工具调用更新的字段，请使用 reducer。

当工具不仅返回数据，而且还改变代理状态时，请使用此功能。

### 错误处理
配置工具错误的处理方式。
[`ToolNode`](https://reference.langchain.com/python/langgraph/agents/#langgraph.prebuilt.tool_node.ToolNode)有关所有选项，请参阅 API 参考文档。

In [ ]:
from langgraph.prebuilt import ToolNode

tools = []

# 默认：捕获调用错误，重新抛出执行错误
tool_node = ToolNode(tools)

# 捕获所有错误，并向 LLM 返回错误信息。
tool_node = ToolNode(tools, handle_tool_errors=True)

# 自定义错误信息
tool_node = ToolNode(tools, handle_tool_errors="Something went wrong, please try again.")

# 自定义错误处理器
def handle_error(e: ValueError) -> str:
    return f"Invalid input: {e}"

tool_node = ToolNode(tools, handle_tool_errors=handle_error)

# 仅捕获特定的异常类型
tool_node = ToolNode(tools, handle_tool_errors=(ValueError, TypeError))

### 使用 tools_condition 进行路由
用于[`tools_condition`](https://reference.langchain.com/python/langgraph/agents/#langgraph.prebuilt.tool_node.tools_condition)根据 LLM 是否发出工具调用进行条件路由：

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.chat_models import init_chat_model

call_llm = init_chat_model()
tools = []

builder = StateGraph(MessagesState)
builder.add_node("llm", call_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)  # Routes to "tools" or END
builder.add_edge("tools", "llm")

graph = builder.compile()

### State Injection 状态注入

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.prebuilt import ToolNode

@tool
def get_message_count(runtime: ToolRuntime) -> str:
    """Get the number of messages in the conversation."""
    messages = runtime.state["messages"]
    return f"There are {len(messages)} messages."

tool_node = ToolNode([get_message_count])